# Module 5: Multi-Mode Optics, Linear Networks & CV Quantum Information
## Notebook 5.2: Two-Photon Interference & The Hong-Ou-Mandel Effect

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/qutip/computational-quantum-optics/blob/main/module_5_multimode_quantum_information/02_hong_ou_mandel_interference.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/qutip/computational-quantum-optics)
[![Python 3.10+](https://img.shields.io/badge/Python-3.10%2B-brightgreen?logo=python)](https://www.python.org/)

---

### Computational Quantum Optics Curriculum
* **Module 0:** Mathematical & Quantum Foundations
* **Module 1:** Single-Mode Quantum States of Light
* **Module 2:** Phase Space Quasiprobability Distributions
* **Module 3:** Light-Matter Interaction & Cavity QED
* **Module 4:** Open Quantum Systems, Dissipation & Trajectories
* **Module 5:** **Multi-Mode Optics, Linear Networks & CV Quantum Information**
  * **5.1** The Quantum Beam Splitter & Entanglement Generation
  * **5.2 Two-Photon Interference & The Hong-Ou-Mandel Effect** $\leftarrow$ *(You are here)*
  * **5.3** Two-Mode Squeezing, EPR States & Continuous-Variable Inseparability
  * **5.4** Continuous-Variable Quantum Teleportation (Braunstein-Kimble)
  * **5.5** Gaussian States, Symplectic Operations & Covariance Matrices
  * **5.6** Quantum State Engineering: Photon Addition, Subtraction & Catalysis
  * **5.7** Linear Optical Quantum Computing & Boson Sampling
* **Module 6:** Advanced Frontiers & Modern Computational Quantum Optics

---

### Learning Objectives
1. **The Quantum Origin of the HOM Effect:** Understand how indistinguishable single photons incident on separate input ports of a 50:50 beam splitter $|1, 1\rangle$ undergo destructive quantum interference, completely canceling coincidence counts.
2. **Path Amplitude Cancellation:** Derive the destructive interference $\mathcal{A}_{\text{trans}} + \mathcal{A}_{\text{refl}} = \frac{1}{2} - \frac{1}{2} = 0$ and verify the generation of the path-entangled NOON state $\frac{|2,0\rangle - |0,2\rangle}{\sqrt{2}}$.
3. **The Temporal Coincidence Dip:** Model Gaussian temporal wavepackets and derive the analytical coincidence probability $P_{11}(\tau) = \frac{1}{2}\left(1 - V e^{-\tau^2/2\sigma^2}\right)$ as a function of delay $\tau$.
4. **Distinguishability & Visibility Degradation:** Quantify the reduction of interference visibility $V = |\langle\psi_a|\psi_b\rangle|^2$ induced by polarization mismatch ($V = \cos^2\theta$), temporal delay, and frequency detuning.
5. **Quantum Beating Fringes:** Simulate quantum beat oscillations inside the HOM envelope when the interfering photons have slightly different central optical frequencies.



In [ ]:
# Environment Setup: Silent Colab detection & high-DPI publication styling
import os
import sys

# Configure headless Agg backend to ensure bulletproof plotting across all environments
if "MPLBACKEND" not in os.environ:
    os.environ["MPLBACKEND"] = "Agg"

import math
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt

# Configure publication matplotlib styling
plt.rcParams.update({
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 10,
    "figure.titlesize": 14,
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "lines.linewidth": 1.8,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "grid.linestyle": "--"
})

# Verify environment capabilities
print(f"NumPy version: {np.__version__}")
try:
    import qutip as qt
    print(f"QuTiP version: {qt.__version__} (Available for optional comparison)")
    HAS_QUTIP = True
except ImportError:
    print("QuTiP not installed; running pure, optimized native NumPy/SciPy engine.")
    HAS_QUTIP = False



---

## Theoretical Framework: The Hong-Ou-Mandel (HOM) Effect

### 1. Two-Photon Scattering on a Balanced Beam Splitter
In 1987, Chung Ki Hong, Zhe-Yu Ou, and Leonard Mandel demonstrated that when two identical single photons arrive simultaneously at the two input ports of a 50:50 beam splitter, **they always exit together in the same output port**. 
Coincidence counts between detectors placed at the two output ports drop strictly to **zero**!

Let mode $a$ and mode $b$ each contain exactly one photon:
$$|\psi_{\text{in}}\rangle = |1, 1\rangle = \hat{a}^\dagger \hat{b}^\dagger |0, 0\rangle$$

Under the balanced 50:50 beam splitter unitary $\hat{U}_{\text{BS}}$:
$$\hat{a}^\dagger \to \frac{\hat{a}^\dagger + \hat{b}^\dagger}{\sqrt{2}}, \quad \hat{b}^\dagger \to \frac{-\hat{a}^\dagger + \hat{b}^\dagger}{\sqrt{2}}$$

Applying this transformation to the two-photon creation operator product:
$$\hat{a}^\dagger \hat{b}^\dagger \to \left(\frac{\hat{a}^\dagger + \hat{b}^\dagger}{\sqrt{2}}\right) \left(\frac{-\hat{a}^\dagger + \hat{b}^\dagger}{\sqrt{2}}\right) = \frac{1}{2} \left[ -(\hat{a}^\dagger)^2 + \hat{a}^\dagger \hat{b}^\dagger - \hat{b}^\dagger \hat{a}^\dagger + (\hat{b}^\dagger)^2 \right]$$

Because bosonic creation operators commute strictly ($[\hat{a}^\dagger, \hat{b}^\dagger] = 0$), the cross-terms cancel identically:
$$\hat{a}^\dagger \hat{b}^\dagger - \hat{b}^\dagger \hat{a}^\dagger \equiv 0!$$

Therefore:
$$\hat{U}_{\text{BS}} |1, 1\rangle = \frac{1}{2}\left[ -(\hat{a}^\dagger)^2 + (\hat{b}^\dagger)^2 \right] |0, 0\rangle = \frac{|0, 2\rangle - |2, 0\rangle}{\sqrt{2}}$$

---

### 2. Physical Origin: Destructive Path Interference
Why is the coincidence state $|1, 1\rangle$ completely missing at the output?
There are two physically indistinguishable ways for a coincidence event to occur:
1. **Both photons are transmitted (TT):** Probability amplitude $\mathcal{A}_{\text{TT}} = t \cdot t = \left(\frac{1}{\sqrt{2}}\right)\left(\frac{1}{\sqrt{2}}\right) = +\frac{1}{2}$.
2. **Both photons are reflected (RR):** Probability amplitude $\mathcal{A}_{\text{RR}} = r_a \cdot r_b = \left(\frac{1}{\sqrt{2}}\right)\left(-\frac{1}{\sqrt{2}}\right) = -\frac{1}{2}$.

Because the photons are identical and indistinguishable, quantum mechanics demands that we **sum probability amplitudes before squaring**:
$$\mathcal{A}_{\text{coincidence}} = \mathcal{A}_{\text{TT}} + \mathcal{A}_{\text{RR}} = \frac{1}{2} - \frac{1}{2} \equiv 0$$
$$P_{11} = |\mathcal{A}_{\text{coincidence}}|^2 = 0!$$

The two photons "bunch" into an entangled two-photon path superposition (an $N=2$ NOON state $\frac{|2, 0\rangle - |0, 2\rangle}{\sqrt{2}}$), with $50\%$ probability of both emerging in port $a$ and $50\%$ probability of both emerging in port $b$.

---

### 3. Temporal Delay & The Hong-Ou-Mandel Coincidence Dip
In practice, single photons are wavepackets with finite temporal duration $\sigma$. 
Let the wavepacket in mode $a$ be delayed by time $\tau$ relative to mode $b$:
$$|\psi_a(t)\rangle = \frac{1}{(2\pi\sigma^2)^{1/4}} e^{-(t - \tau)^2 / (4\sigma^2)}, \quad |\psi_b(t)\rangle = \frac{1}{(2\pi\sigma^2)^{1/4}} e^{-t^2 / (4\sigma^2)}$$

The quantum overlap between the two temporal modes is:
$$\xi(\tau) = \int_{-\infty}^{\infty} dt \, \psi_a^*(t) \psi_b(t) = \exp\left(-\frac{\tau^2}{8\sigma^2}\right)$$

The coincidence probability recorded by slow detectors integrating over time is:
$$P_{11}(\tau) = \frac{1}{2} \left[ 1 - V |\xi(\tau)|^2 \right] = \frac{1}{2} \left[ 1 - V \exp\left(-\frac{\tau^2}{4\sigma^2}\right) \right]$$
where $V \in [0, 1]$ is the **HOM visibility**:
$$V = \frac{P_{11}(\infty) - P_{11}(0)}{P_{11}(\infty)}$$

* **Zero Delay ($\tau = 0$):** $P_{11}(0) = \frac{1}{2}(1 - V) = 0$ for ideal indistinguishable photons ($V = 100\%$).
* **Large Delay ($|\tau| \gg \sigma$):** The photons arrive at different times and are completely distinguishable; no interference occurs. Each photon independently flips a classical coin with $50\%$ transmission:
  $$P_{11}(\infty) = P(T_a)P(R_b) + P(R_a)P(T_b) = \frac{1}{2}\cdot\frac{1}{2} + \frac{1}{2}\cdot\frac{1}{2} = \frac{1}{2}$$

---

### 4. Distinguishability: Polarization Mismatch & Spectral Beating
If the photons are distinguishable by internal degrees of freedom:
1. **Polarization Mismatch:** If photon $a$ is polarized at angle $\theta$ relative to photon $b$:
   $$|\langle \vec{\epsilon}_a | \vec{\epsilon}_b \rangle|^2 = \cos^2\theta \implies V(\theta) = \cos^2\theta$$
2. **Frequency Detuning $\Delta\omega$:** If the photons have different central frequencies $\omega_a - \omega_b = \Delta\omega$:
   $$P_{11}(\tau) = \frac{1}{2}\left[ 1 - \exp\left(-\frac{\tau^2}{4\sigma^2}\right) \cos(\Delta\omega \tau) \right]$$
   The coincidence dip is modulated by **quantum beating fringes** with spatial frequency $\Delta\omega$.



In [ ]:
# Implementation: High-Precision Two-Photon HOM Interference Engine

class HongOuMandelEngine:
    """
    Simulation engine for two-photon Hong-Ou-Mandel interference.
    Models Fock-basis scattering, temporal wavepacket overlaps, polarization mismatch, and frequency detuning.
    """
    def __init__(self, sigma_t=1.0):
        self.sigma = float(sigma_t)  # Pulse duration / coherence time

    def coincidence_probability_delay(self, tau_array, visibility=1.0):
        """
        Compute coincidence detection probability P_11(tau) vs. relative delay tau:
            P_11(tau) = 0.5 * (1 - V * exp(-tau^2 / (4 * sigma^2)))
        """
        tau = np.asarray(tau_array, dtype=float)
        V = float(visibility)
        overlap_sq = np.exp(-(tau**2) / (4.0 * self.sigma**2))
        return 0.5 * (1.0 - V * overlap_sq)

    def coincidence_with_spectral_beating(self, tau_array, delta_omega=0.0):
        """
        Compute coincidence probability P_11(tau) with frequency detuning delta_omega:
            P_11(tau) = 0.5 * [1 - exp(-tau^2 / (4 * sigma^2)) * cos(delta_omega * tau)]
        """
        tau = np.asarray(tau_array, dtype=float)
        d_omega = float(delta_omega)
        envelope = np.exp(-(tau**2) / (4.0 * self.sigma**2))
        return 0.5 * (1.0 - envelope * np.cos(d_omega * tau))

    def fock_state_hom_scattering(self, n_a=1, n_b=1):
        """
        Exact numerical Fock-basis scattering on a 50:50 beam splitter for |n_a, n_b> input.
        Returns the output Fock state amplitudes in basis |k, n_tot - k>.
        """
        n_tot = n_a + n_b
        out_amplitudes = np.zeros(n_tot + 1, dtype=complex)
        
        # Beam splitter matrix: [[1, 1], [-1, 1]] / sqrt(2)
        # Using polynomial expansion of (a^dag + b^dag)^n_a * (-a^dag + b^dag)^n_b
        # via binomial expansion
        sqrt2 = np.sqrt(2.0)
        
        for k_a in range(n_a + 1):
            c_a = math.comb(n_a, k_a)
            # from (a^dag + b^dag)^n_a: k_a terms of a^dag, (n_a - k_a) terms of b^dag
            for k_b in range(n_b + 1):
                c_b = math.comb(n_b, k_b)
                # from (-a^dag + b^dag)^n_b: k_b terms of -a^dag, (n_b - k_b) terms of b^dag
                total_a_photons = k_a + k_b
                sign = (-1.0)**k_b
                
                weight = (c_a * c_b * sign) / (sqrt2**n_tot)
                out_amplitudes[total_a_photons] += weight
                
        # Normalize with Fock factor sqrt(k! (n_tot - k)! / (n_a! n_b!))
        fock_norm = np.zeros(n_tot + 1, dtype=float)
        for k in range(n_tot + 1):
            num = math.factorial(k) * math.factorial(n_tot - k)
            denom = math.factorial(n_a) * math.factorial(n_b)
            fock_norm[k] = np.sqrt(num / denom)
            
        out_amplitudes *= fock_norm
        probs = np.abs(out_amplitudes)**2
        
        return {
            "n_tot": n_tot,
            "amplitudes": out_amplitudes,
            "probabilities": probs
        }

print("HongOuMandelEngine successfully loaded.")



In [ ]:
# Publication-Quality 4-Panel Visualization: Hong-Ou-Mandel Interference & Quantum Beating

hom = HongOuMandelEngine(sigma_t=1.0)

# Time delay scan
tau_grid = np.linspace(-6.0, 6.0, 1001)

# 1. Canonical HOM dip for various visibilities V = 1.0, 0.75, 0.5, 0.0
P_ideal = hom.coincidence_probability_delay(tau_grid, visibility=1.0)
P_75 = hom.coincidence_probability_delay(tau_grid, visibility=0.75)
P_50 = hom.coincidence_probability_delay(tau_grid, visibility=0.5)
P_class = hom.coincidence_probability_delay(tau_grid, visibility=0.0)

# 2. Output Fock probabilities for |1, 1> on 50:50 BS
fock_res_11 = hom.fock_state_hom_scattering(1, 1)

# 3. Spectral beating inside HOM envelope: delta_omega * sigma = 0, 2.5, 5.0
P_beat_0 = hom.coincidence_with_spectral_beating(tau_grid, delta_omega=0.0)
P_beat_1 = hom.coincidence_with_spectral_beating(tau_grid, delta_omega=2.5)
P_beat_2 = hom.coincidence_with_spectral_beating(tau_grid, delta_omega=5.0)

# 4. Visibility vs. polarization angle mismatch theta
theta_polarization = np.linspace(0.0, np.pi / 2.0, 100)
V_polarization = np.cos(theta_polarization)**2

# Generate 4-Panel Masterpiece Figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# -------------------------------------------------------------
# Panel (a): Canonical Hong-Ou-Mandel Coincidence Dip
# -------------------------------------------------------------
ax_a = axes[0, 0]
ax_a.plot(tau_grid, P_ideal, color="#e74c3c", linewidth=2.4, label=r"Ideal Indistinguishable ($V = 100\%$)")
ax_a.plot(tau_grid, P_75, color="#e67e22", linewidth=2.0, label=r"Partial Overlap ($V = 75\%$)")
ax_a.plot(tau_grid, P_50, color="#3498db", linewidth=2.0, label=r"Distinguishable ($V = 50\%$)")
ax_a.plot(tau_grid, P_class, color="black", linestyle="--", linewidth=1.8, label=r"Classical Limit $P_{11} \equiv 0.5$ ($V=0$)")

ax_a.scatter([0.0], [0.0], color="#e74c3c", marker="o", s=80, zorder=5)
ax_a.annotate("Destructive Interference:\n$P_{11}(0) = 0$ (Zero Coincidences)", 
              xy=(0.0, 0.0), xytext=(0.8, 0.15),
              arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#fadbd8", ec="#e74c3c", lw=1.2),
              fontweight="bold")

ax_a.set_title(r"(a) Hong-Ou-Mandel Coincidence Dip $P_{11}(\tau)$", fontweight="bold")
ax_a.set_xlabel(r"Relative Delay $\tau / \sigma$")
ax_a.set_ylabel(r"Coincidence Probability $P_{11}(\tau)$")
ax_a.set_ylim(-0.03, 0.55)
ax_a.legend(loc="lower right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (b): Output State Fock Distribution: Photon Bunching
# -------------------------------------------------------------
ax_b = axes[0, 1]
fock_states = [r"$|2, 0\rangle$", r"$|1, 1\rangle$", r"$|0, 2\rangle$"]
probs_11 = fock_res_11["probabilities"]  # probabilities for n_a = 0, 1, 2
# Note: n_a = 0 -> |0, 2>; n_a = 1 -> |1, 1>; n_a = 2 -> |2, 0>
bar_probs = [probs_11[2], probs_11[1], probs_11[0]]
colors_bar = ["#8e44ad", "#e74c3c", "#8e44ad"]

bars = ax_b.bar(fock_states, bar_probs, color=colors_bar, width=0.45, alpha=0.85, edgecolor="black")
ax_b.set_ylim(0.0, 0.65)

# Value annotations
for bar in bars:
    yval = bar.get_height()
    ax_b.text(bar.get_x() + bar.get_width()/2.0, yval + 0.02, f"{yval:.2f}", ha='center', va='bottom', fontweight='bold', fontsize=11)

ax_b.annotate("Coincidence Extinguished:\n" + r"$P(1, 1) \equiv 0$", xy=(1.0, 0.0), xytext=(0.6, 0.28),
              arrowprops=dict(arrowstyle="->", color="#e74c3c", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#fff9c4", ec="#f39c12", lw=1.2),
              fontweight="bold")

ax_b.set_title(r"(b) Output State for $|1, 1\rangle$ Input (NOON State)", fontweight="bold")
ax_b.set_ylabel(r"Probability $P(n_a, n_b)$")

# -------------------------------------------------------------
# Panel (c): Spectral Beating inside the HOM Dip
# -------------------------------------------------------------
ax_c = axes[1, 0]
ax_c.plot(tau_grid, P_beat_0, color="#7f8c8d", linestyle="--", linewidth=1.8, label=r"Degenerate $\Delta\omega = 0$")
ax_c.plot(tau_grid, P_beat_1, color="#2980b9", linewidth=2.0, label=r"Detuned $\Delta\omega\sigma = 2.5$")
ax_c.plot(tau_grid, P_beat_2, color="#16a085", linewidth=2.2, label=r"Detuned $\Delta\omega\sigma = 5.0$")

ax_c.set_title(r"(c) Spectral Quantum Beating inside HOM Envelope", fontweight="bold")
ax_c.set_xlabel(r"Relative Delay $\tau / \sigma$")
ax_c.set_ylabel(r"Coincidence Probability $P_{11}(\tau)$")
ax_c.set_ylim(-0.05, 0.65)
ax_c.legend(loc="lower right", framealpha=0.9)

# -------------------------------------------------------------
# Panel (d): Visibility vs. Polarization Angle Mismatch
# -------------------------------------------------------------
ax_d = axes[1, 1]
ax_d.plot(theta_polarization * 180.0 / np.pi, V_polarization, color="#8e44ad", linewidth=2.4, label=r"Theory $V = \cos^2\theta$")
ax_d.scatter([0.0, 30.0, 45.0, 60.0, 90.0], [1.0, 0.75, 0.5, 0.25, 0.0], color="#e74c3c", s=70, zorder=5, label="Benchmark Angles")

ax_d.annotate(r"Orthogonal Polarization:" + "\n" + r"$V(90^\circ) = 0$ (Classical Coins)", 
              xy=(90.0, 0.0), xytext=(45.0, 0.15),
              arrowprops=dict(arrowstyle="->", color="black", lw=1.5),
              bbox=dict(boxstyle="round,pad=0.4", fc="#f4ecf7", ec="#8e44ad", lw=1.2),
              fontweight="bold")

ax_d.set_title(r"(d) HOM Interference Visibility vs. Polarization Angle", fontweight="bold")
ax_d.set_xlabel(r"Relative Polarization Angle $\theta$ (degrees)")
ax_d.set_ylabel(r"Interference Visibility $V$")
ax_d.set_ylim(-0.05, 1.08)
ax_d.legend(loc="upper right", framealpha=0.9)

plt.tight_layout()
plt.show()



> **Physical Insight: Hong-Ou-Mandel Interference as a Purely Quantum Phenomenon**
>
> 1. **Why Doesn't Classical Light Show an HOM Dip to Zero?** For classical electromagnetic wavepackets with intensity $I_a(t)$ and $I_b(t)$, intensity correlations measured by square-law detectors are bounded by the classical Cauchy-Schwarz inequality. The classical coincidence probability between two independent beams cannot drop below $P_{11}(0) \ge \frac{1}{2} P_{11}(\infty)$, meaning **classical wave interference can never achieve a visibility exceeding $V_{\text{classical}} \le 50\%$**! Observing $V > 50\%$ (and ideally $V = 100\%$) is an unequivocal signature of single-photon quantum interference.
> 2. **Indistinguishability as the Currency of Quantum Information:** The HOM dip depth measures the degree of indistinguishability between two single-photon wavepackets across all degrees of freedom: spatial, temporal, spectral, and polarization. In linear optical quantum computing (KLM protocol) and photonic quantum networks, two-photon HOM interference is the fundamental building block of entangling CNOT and CZ quantum logic gates.
> 3. **Feynman's Principle of Indistinguishable Paths:** When the photons arrive with delay $|\tau| \gg \sigma$, the arrival times at the detectors reveal which photon took which path (TT vs. RR), destroying quantum interference. When $\tau = 0$, it is fundamentally impossible—even in principle—to know which photon went where, forcing the probability amplitudes $\mathcal{A}_{\text{TT}}$ and $\mathcal{A}_{\text{RR}}$ to interfere.



In [ ]:
# Interactive Exploration: Hong-Ou-Mandel Delay & Distinguishability Tuner

def interactive_hom_explorer(visibility=1.0, delta_omega=0.0, delay_tau=0.0):
    """
    Interactive exploration of the HOM coincidence dip and spectral beating.
    """
    hom_engine = HongOuMandelEngine(sigma_t=1.0)
    tau_scan = np.linspace(-5.0, 5.0, 501)
    
    # Coincidence curve
    if delta_omega == 0.0:
        P_curve = hom_engine.coincidence_probability_delay(tau_scan, visibility=visibility)
    else:
        P_curve = 0.5 * (1.0 - visibility * np.exp(-(tau_scan**2)/4.0) * np.cos(delta_omega * tau_scan))
        
    P_current = 0.5 * (1.0 - visibility * np.exp(-(delay_tau**2)/4.0) * np.cos(delta_omega * delay_tau))
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(tau_scan, P_curve, color="#e74c3c", linewidth=2.2, label="HOM Coincidence $P_{11}(\\tau)$")
    ax.axhline(0.5, color="black", linestyle="--", alpha=0.5, label="Classical Limit (0.5)")
    ax.scatter([delay_tau], [P_current], color="#2980b9", s=100, zorder=5, label=f"Current Delay (P_11 = {P_current:.3f})")
    
    ax.set_title(f"HOM Coincidence Dip (Visibility $V = {visibility*100:.1f}\\%$, $\\Delta\\omega = {delta_omega:.1f}$)")
    ax.set_xlabel(r"Relative Delay $\tau / \sigma$")
    ax.set_ylabel(r"Coincidence Probability $P_{11}$")
    ax.set_ylim(-0.05, 0.6)
    ax.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

# Test invocation
interactive_hom_explorer(visibility=0.9, delta_omega=1.5, delay_tau=0.0)

try:
    import ipywidgets as widgets
    from IPython.display import display
    w = widgets.interactive(
        interactive_hom_explorer,
        visibility=widgets.FloatSlider(value=1.0, min=0.0, max=1.0, step=0.05, description="Visibility V:"),
        delta_omega=widgets.FloatSlider(value=0.0, min=0.0, max=5.0, step=0.5, description=r"$\Delta\omega\sigma$:"),
        delay_tau=widgets.FloatSlider(value=0.0, min=-4.0, max=4.0, step=0.2, description=r"Delay $\tau$:")
    )
    display(w)
except Exception:
    pass



---

## Guided Computational Challenges & Exercises

### Challenge 1: Perfect Destructive Interference Benchmark
1. In the Fock basis, prove algebraically that for an input $|1, 1\rangle$ on a 50:50 beam splitter, the coincidence probability is identically zero:
   $$P(1, 1) \equiv 0.0$$
2. Assert numerically using `fock_state_hom_scattering(1, 1)` that $P(1, 1) < 10^{-15}$.
3. Assert that the bunched state probabilities satisfy $P(2, 0) = P(0, 2) = 0.5 \pm 10^{-15}$.

---

### Challenge 2: Coincidence Dip Integral & Full Width at Half Minimum (FWHM)
1. For Gaussian pulses with temporal width $\sigma$, the coincidence dip profile is:
   $$P_{11}(\tau) = \frac{1}{2}\left(1 - e^{-\tau^2 / 4\sigma^2}\right)$$
2. Show analytically that the Full Width at Half Minimum (FWHM) of the dip is:
   $$\text{FWHM} = 4\sigma \sqrt{\ln 2} \approx 3.3302 \sigma$$
3. Numerically evaluate the half-minimum crossing points of $P_{11}(\tau)$ and assert that the empirical FWHM matches $4\sigma\sqrt{\ln 2}$ to within $0.05\%$.

---

### Challenge 3: Higher-Order Multi-Photon Bunching ($|2, 2\rangle$ Input)
1. Compute the output state when two pairs of photons $|2, 2\rangle$ enter the beam splitter:
   $$\hat{U}_{\text{BS}}|2, 2\rangle = \frac{\sqrt{6}}{4}|4, 0\rangle - \frac{1}{2}|2, 2\rangle + \frac{\sqrt{6}}{4}|0, 4\rangle$$
2. Show that unlike the single-photon case, coincidence counts do **not** vanish: $P(2, 2) = \frac{1}{4} = 0.25$.
3. Assert this result numerically to machine precision.



In [ ]:
# Numerical Verification of Challenges 1, 2, and 3: Hong-Ou-Mandel Benchmarks

hom_engine = HongOuMandelEngine(sigma_t=1.0)

# -------------------------------------------------------------
# Part 1: Perfect Destructive Interference for |1, 1> (Challenge 1)
# -------------------------------------------------------------
res_11 = hom_engine.fock_state_hom_scattering(1, 1)
p_20 = res_11["probabilities"][2]  # n_a = 2, n_b = 0
p_11 = res_11["probabilities"][1]  # n_a = 1, n_b = 1
p_02 = res_11["probabilities"][0]  # n_a = 0, n_b = 2

print(f"Fock |1, 1> HOM Scattering Check:")
print(f"- P(2, 0): {p_20:.12f} (Expected: 0.5)")
print(f"- P(1, 1): {p_11:.2e} (Expected: 0.0)")
print(f"- P(0, 2): {p_02:.12f} (Expected: 0.5)")

assert p_11 < 1e-15, f"HOM destructive interference violated: P(1,1) = {p_11:.2e} >= 1e-15!"
assert np.isclose(p_20, 0.5, atol=1e-15), f"P(2,0) mismatch: {p_20} != 0.5!"
assert np.isclose(p_02, 0.5, atol=1e-15), f"P(0,2) mismatch: {p_02} != 0.5!"

# -------------------------------------------------------------
# Part 2: HOM Dip FWHM Benchmark (Challenge 2)
# -------------------------------------------------------------
tau_dense = np.linspace(-5.0, 5.0, 100000)
P_dense = hom_engine.coincidence_probability_delay(tau_dense, visibility=1.0)

# Dip minimum is at 0 (P=0.0), baseline is at 0.5. Half-minimum is at P = 0.25
half_max_level = 0.25
indices_below_half = np.where(P_dense <= half_max_level)[0]

tau_left = tau_dense[indices_below_half[0]]
tau_right = tau_dense[indices_below_half[-1]]
fwhm_emp = tau_right - tau_left
fwhm_analytic = 4.0 * hom_engine.sigma * np.sqrt(np.log(2.0))

print(f"\nHOM Dip FWHM Benchmark (sigma = {hom_engine.sigma}):")
print(f"- Empirical FWHM:  {fwhm_emp:.6f}")
print(f"- Analytical FWHM: {fwhm_analytic:.6f}")
print(f"- Discrepancy:     {np.abs(fwhm_emp - fwhm_analytic):.2e}")

assert np.isclose(fwhm_emp, fwhm_analytic, rtol=5e-4), "HOM dip FWHM deviates by > 0.05%!"

# -------------------------------------------------------------
# Part 3: Higher-Order Fock |2, 2> Scattering Check (Challenge 3)
# -------------------------------------------------------------
res_22 = hom_engine.fock_state_hom_scattering(2, 2)
# Output probabilities for n_a = 0, 1, 2, 3, 4
p_40 = res_22["probabilities"][4]
p_22 = res_22["probabilities"][2]
p_04 = res_22["probabilities"][0]

expected_p_22 = 0.25  # (1/2)^2 = 1/4
expected_p_40 = 6.0 / 16.0  # (sqrt(6)/4)^2 = 6/16 = 3/8 = 0.375

print(f"\nHigher-Order Fock |2, 2> Scattering Benchmark:")
print(f"- P(2, 2): {p_22:.6f} (Expected: 0.25)")
print(f"- P(4, 0): {p_40:.6f} (Expected: 0.375)")
print(f"- P(0, 4): {p_04:.6f} (Expected: 0.375)")

assert np.isclose(p_22, expected_p_22, atol=1e-14), f"|2, 2> coincidence mismatch: {p_22} != 0.25!"
assert np.isclose(p_40, expected_p_40, atol=1e-14), f"|2, 2> P(4,0) mismatch: {p_40} != 0.375!"
assert np.isclose(p_04, expected_p_40, atol=1e-14), f"|2, 2> P(0,4) mismatch: {p_04} != 0.375!"

print("=" * 70)
print("SUCCESS: Two-Photon HOM Interference Numerically Verified & Benchmarked!")
print(f"- Single-photon HOM cancellation: P(1,1) = {p_11:.2e} == 0 (PASS)")
print(f"- NOON state bunching:           P(2,0) = {p_20:.2f}, P(0,2) = {p_02:.2f} (PASS)")
print(f"- Coincidence dip FWHM:          {fwhm_emp:.4f} vs. {fwhm_analytic:.4f} (PASS: < 0.05%)")
print(f"- Multi-photon |2,2> scattering: P(2,2) = {p_22:.4f} == 0.25 (PASS)")
print("=" * 70)



---

## Summary & Next Steps

### Key Takeaways from Notebook 5.2
1. **Destructive Quantum Interference:** Two indistinguishable photons entering a 50:50 beam splitter $|1, 1\rangle$ undergo complete destructive path cancellation: $\mathcal{A}_{\text{TT}} + \mathcal{A}_{\text{RR}} = \frac{1}{2} - \frac{1}{2} = 0$.
2. **NOON State Generation:** The output state collapses into the entangled bunching state $\frac{|2, 0\rangle - |0, 2\rangle}{\sqrt{2}}$, with zero probability of detecting one photon in each output port.
3. **The HOM Dip:** Introducing a relative delay $\tau$ reveals the wavepacket coherence envelope: $P_{11}(\tau) = \frac{1}{2}(1 - V e^{-\tau^2/4\sigma^2})$ with FWHM $\approx 3.33\sigma$.
4. **Indistinguishability as a Resource:** Any distinguishing degree of freedom (polarization mismatch, spectral detuning, temporal jitter) degrades the interference visibility ($V < 100\%$).

---

### Up Next: Notebook 5.3
In [Notebook 5.3: Two-Mode Squeezing, EPR States & Continuous-Variable Inseparability](03_two_mode_squeezing_and_epr_states.ipynb), we explore continuous-variable bipartite entanglement: generating **Two-Mode Squeezed Vacuum (TMSV)**, demonstrating the EPR paradox with relative position and momentum variances, and computing the Duan-Giedke-Cirac-Zoller (DGCZ) inseparability criterion.

